## 微调模型 DeepSeek R1（推理模型）蒸馏至llama8b。使用medicaal -cot数据集前一万条。
原DAATACCAMP网站https://www.datacamp.com/tutorial/fine-tuning-deepseek-r1-reasoning-model

在本教程中，我们将在 Hugging Face 的医疗思路链数据集上对模型进行微调DeepSeek-R1-Distill-Llama-8B。这个精简的 DeepSeek-R1 模型是通过在使用 DeepSeek-R1 生成的数据上对 Llama 3.1 8B 模型进行微调而创建的。它展示了与原始模型类似的推理能力。

数据集中文版本：https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT-zh

模型下载地址：https://huggingface.co/unsloth/DeepSeek-R1-Distill-Llama-8B

In [ ]:
#%%capture
#用于捕获单元格的输出；想要暂时隐藏输出，使笔记本更整洁时
#!pip install unsloth
#安装unsloth

In [ ]:
from huggingface_hub import login
import wandb
HUGGINGFACE_TOKEN ="*****************************"
WANDB_TOKEN ="*********************************"

hf_token =HUGGINGFACE_TOKEN
login(hf_token)

wb_token =WANDB_TOKEN

wandb.login(key=wb_token)
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-Distill-Llama-8B on Medical COT Dataset', 
    job_type="training", 
    anonymous="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/aeon/.netrc
wandb: Currently logged in as: aeona (MIPS-LAB) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


In [3]:
''' 
创建模型，创建tokenizer,使用unsloth优化的FastLanguageModel来创建
'''
from unsloth import FastLanguageModel

max_seq_length = 2048 
dtype = None 
load_in_4bit = True


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token, 
)

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-20 18:59:49 __init__.py:190] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.12: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA RTX A6000. Max memory: 47.536 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [4]:
'''
为了为模型创建提示样式，我们将定义一个系统提示，并包含用于生成问题和响应的占位符。提示将引导模型逐步思考并提供合乎逻辑且准确的响应。

'''

prompt_style = """以下是描述任务的指示，并附有提供更多上下文的输入内容。
请写出恰当完成该请求的回答。
在回答之前，请仔细思考问题，并创建逐步的思维链，以确保回答合乎逻辑且准确.

### Instruction:
你是一位在临床推理、诊断和治疗计划方面具有专业知识的医学专家。
请回答以下医学问题. 

### Question:
{}

### Response:
<think>{}"""

# 测试用医学问题
question = "一位61岁的女性,长期存在咳嗽或打喷嚏等活动时不自主尿失禁的病史,但夜间无漏尿。她接受了妇科检查和Q-tip测试。基于这些发现,膀胱测压最可能显示她的残余尿量和逼尿肌收缩情况如何?"

# 设置模型为推理模式
FastLanguageModel.for_inference(model) 
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 生成回答
outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print("### 微调前模型推理结果：")
print(response[0].split("### Response:")[1])

### 微调前模型推理结果：

<think>
好的，我现在需要解决一个关于膀胱功能的问题。让我仔细分析一下问题。

首先，患者是一个61岁的女性，有长期咳嗽或打喷嚏时不自主的尿失禁病史，但夜间没有漏尿。她的妇科检查和Q-tip测试结果显示出膀胱功能异常。现在需要确定膀胱测压显示她的残余尿量和逼尿肌收缩情况如何。

我回想起膀胱功能的基本概念。膀胱测压主要用于评估膀胱的残余尿量和逼尿肌的功能。测压通常在尿动力学研究中使用，通过检查膀胱内的压力变化来判断膀胱的充盈能力和排尿肌的状态。

患者的咳嗽或打喷嚏导致不自主尿失禁，这可能是由于膀胱的不完全充盈，导致膀胱内压力不足，无法维持尿道口的闭合。然而，她夜间没有漏尿，这可能意味着她在夜间能够通过膀胱的自我控制机制排出足够的尿液，或者她的膀胱在夜间能够保持一定的压力。

接下来，考虑到Q-tip测试的结果。Q-tip测试通常用于评估膀胱的充盈能力和排尿肌的功能。Q-tip测量的是膀胱颈部的位置和压力。当膀胱充盈时，Q-tip应位于肛门旁边的位置。如果Q-tip位于腹股沟的位置，可能说明膀胱充盈能力较差，膀胱内压力不足，导致尿失禁。

在这种情况下，患者的残余尿量可能较多，因为膀胱的充盈能力不足，无法有效地储存足够的尿液。随着咳嗽或打喷嚏的触发，膀胱内压力迅速下降，导致尿失禁。然而，夜间没有漏尿，可能是因为夜间排尿次数较多，或者她的膀胱在夜间能够暂时维持一定的压力。

至于逼尿肌收缩情况，患者的不自主尿失禁可能是因为膀胱内压力不足，导致排尿肌无法有效地抑制尿流。膀胱测压可能显示出膀胱内压力较低，膀胱的充盈能力较差，排尿肌的收缩力不足以维持尿道口闭合。

综合以上分析，膀胱测压可能显示患者的残余尿量较多，膀胱内压力较低，膀胱充盈能力不足，排尿肌收缩力较弱，导致不自主尿失禁。
</think>

基于患者的病史和检查结果，膀胱测压可能显示以下情况：

1. **残余尿量**：较多。由于膀胱的充盈能力不足，膀胱无法有效储存足够的尿液，容易在咳嗽或打喷嚏时失禁。

2. **逼尿肌收缩情况**：收缩力较弱。膀胱内压力不足，排尿肌无法有效抑制尿流，导致不自主尿失禁。

总结来说，膀胱测压可能显示患者有较多的残余尿量和较弱的逼尿肌收缩能力，导致不自主尿失禁的情况。<｜end▁of▁sentence｜>


In [5]:
train_prompt_style = """以下是描述任务的指令，以及提供更多上下文的输入。
                        请写出恰当完成该请求的回答。
                        在回答之前，请仔细思考问题，并创建一个逐步的思维链，以确保回答合乎逻辑且准确。

                        ### Instruction:
                        你是一位在临床推理、诊断和治疗计划方面具有专业知识的医学专家。
                        请回答以下医学问题。

                        ### Question:
                        {}

                        ### Response:
                        <think>
                        {}
                        </think>
                        {}"""

EOS_TOKEN = tokenizer.eos_token  # 添加结束符标记

#格式化提示函数,用于处理数据集中的示例
def formatting_prompts_func(examples):
    # 从examples中提取问题、思维链和回答
    inputs = examples["Question"]      # 医学问题列表
    cots = examples["Complex_CoT"]     # 思维链列表 
    outputs = examples["Response"]     # 回答列表
    
    # 存储格式化后的文本
    texts = []
    
    # 遍历每个示例,将问题、思维链和回答组合成指定格式
    for input, cot, output in zip(inputs, cots, outputs):
        # 使用train_prompt_style模板格式化文本,并添加结束符
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
        
    # 返回格式化后的文本字典
    return {
        "text": texts,
    }

# 加载数据集并应用格式化
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","zh", split = "train[0:10000]",trust_remote_code=True)
dataset = dataset.map(formatting_prompts_func, batched = True,)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [6]:

"""
配置LoRA微调参数
使用LoRA技术进行参数高效微调
"""
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    # 原始模型
    model, 
    # LoRA秩,用于控制低秩矩阵的维度,值越大表示可训练参数越多,模型性能可能更好但训练开销更大
    # 建议: 8-32之间
    r=16,  
    # 需要应用LoRA的目标模块列表
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention相关层
        "gate_proj", "up_proj", "down_proj",     # FFN相关层
    ],
    # LoRA缩放因子,用于控制LoRA更新的幅度。值越大，LoRA的更新影响越大。
    lora_alpha=16,
    # LoRA层的dropout率,用于防止过拟合,这里设为0表示不使用dropout。
    # 如果数据集较小，建议设置0.1左右。
    lora_dropout=0,  
    # 是否对bias参数进行微调,none表示不微调bias
    # none: 不微调偏置参数；
    # all: 微调所有参数；
    # lora_only: 只微调LoRA参数。
    bias="none",  
    # 是否使用梯度检查点技术节省显存,使用unsloth优化版本
    # 会略微降低训练速度，但可以显著减少显存使用
    use_gradient_checkpointing="unsloth", 
    # 随机数种子,用于结果复现
    random_state=3407,
    # 是否使用rank-stabilized LoRA,这里不使用
    # 会略微降低训练速度，但可以显著减少显存使用
    use_rslora=False,  
    # LoFTQ配置,这里不使用该量化技术，用于进一步压缩模型大小
    loftq_config=None,
)
"""
配置训练参数和初始化训练器
"""
from trl import SFTTrainer  # 用于监督微调的训练器
from transformers import TrainingArguments  # 用于配置训练参数
from unsloth import is_bfloat16_supported  # 检查是否支持bfloat16精度训练

# 初始化SFT训练器
trainer = SFTTrainer(
    model=model,  # 待训练的模型
    tokenizer=tokenizer,  # 分词器
    train_dataset=dataset,  # 训练数据集
    dataset_text_field="text",  # 数据集字段的名称
    max_seq_length=max_seq_length,  # 最大序列长度
    dataset_num_proc=2,  # 数据集处理的并行进程数，提高CPU利用率
    args=TrainingArguments(
        per_device_train_batch_size=2,  # 每个GPU的训练批次大小
        gradient_accumulation_steps=4,   # 梯度累积步数,用于模拟更大的batch size
        warmup_steps=5,  # 预热步数,逐步增加学习率
        learning_rate=2e-4,  # 学习率
        lr_scheduler_type="linear",  # 线性学习率调度器
        max_steps=60,    # 最大训练步数（一步 = 处理一个batch的数据）
        # 根据硬件支持选择训练精度
        fp16=not is_bfloat16_supported(),  # 如果不支持bf16则使用fp16
        bf16=is_bfloat16_supported(),      # 如果支持则使用bf16
        logging_steps=10,  # 每10步记录一次日志
        optim="adamw_8bit",  # 使用8位AdamW优化器节省显存，几乎不影响训练效果
        weight_decay=0.01,   # 权重衰减系数,用于正则化，防止过拟合
        seed=3407,  # 随机数种子
        output_dir="outputs",  # 保存模型检查点和训练日志
    ),
)

Unsloth 2025.2.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Applying chat template to train dataset (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
"""
开始训练
"""
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 41,943,040
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
10,2.051400
20,1.629100
30,1.519800
40,1.424600
50,1.477700
60,1.385400


TrainOutput(global_step=60, training_loss=1.581319014231364, metrics={'train_runtime': 320.8099, 'train_samples_per_second': 1.496, 'train_steps_per_second': 0.187, 'total_flos': 2.007650954964173e+16, 'train_loss': 1.581319014231364})

In [8]:
"""
微调后的模型推理测试
"""
question = "一位61岁的女性,长期存在咳嗽或打喷嚏等活动时不自主尿失禁的病史,但夜间无漏尿。她接受了妇科检查和Q-tip测试。基于这些发现,膀胱测压最可能显示她的残余尿量和逼尿肌收缩情况如何?"

# 启用模型推理模式,使用Unsloth加速推理速度
FastLanguageModel.for_inference(model)  

# 对输入问题进行编码,转换为模型可处理的张量格式并移至GPU
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# 生成回答
outputs = model.generate(
    input_ids=inputs.input_ids, # 输入token的id序列
    attention_mask=inputs.attention_mask,  # 注意力掩码,用于标记有效输入位置
    max_new_tokens=1200, # 生成的最大新token数量
    use_cache=True, # 是否使用KV缓存加速生成
)

# 解码模型输出
response = tokenizer.batch_decode(outputs)
print("### 微调后模型推理结果：")
print(response[0].split("### Response:")[1])

### 微调后模型推理结果：

<think>
嗯，这位61岁的女性有咳嗽或打喷嚏时不自主尿失禁的病史，但夜间没漏尿。听起来好像是膀胱反射活动异常。嗯，她做了妇科检查和Q-tip测试，应该是针对膀胱反射的问题。

首先，Q-tip测试是用来评估膀胱反射功能的常用方法。Q-tip放入膀胱后，如果膀胱测压高，说明膀胱反射功能可能较强，尿量少。测压低的话，可能膀胱反射功能弱，尿量多。

根据她的病史，她没有夜间漏尿，说明夜间的膀胱反射可能正常，尿量控制没问题。那么，可能她的膀胱反射功能在白天不太好，容易漏尿。

在Q-tip测试中，如果测压高，膀胱反射功能强，尿量少，可能在白天容易漏尿。测压低的话，膀胱反射功能弱，尿量多，可能在白天容易漏尿。

所以，结合这些信息，假设她的Q-tip测压高，膀胱反射功能强，尿量少，可能在白天容易漏尿。虽然她没有夜间漏尿，但白天可能还是容易漏尿。

嗯，总结一下，可能她的膀胱测压高，膀胱反射功能强，尿量少，容易在白天漏尿。
</think>
根据题目中提供的信息和上述推理，膀胱测压高，膀胱反射功能强，尿量少，容易在白天漏尿。<｜end▁of▁sentence｜>


In [9]:
"""
保存模型
包括保存完整模型和合并后的模型
"""
new_model_local = "DeepSeek-R1-Medical-COT-Qwen-7B"
model.save_pretrained(new_model_local) 
tokenizer.save_pretrained(new_model_local)

# 保存合并后的16bit模型
model.save_pretrained_merged(new_model_local, tokenizer, save_method = "merged_16bit",)



Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 31.51 out of 62.54 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [00:01<00:00, 17.04it/s]


Unsloth: Saving tokenizer... Done.
Done.


In [ ]:
"""
模型上传代码
"""
# 定义在线仓库地址
new_model_online = "AEONA/DeepSeek-R1-Medical-COT-Qwen-7B"
# 上传LoRA权重和配置
model.push_to_hub(new_model_online)
# 上传分词器    
tokenizer.push_to_hub(new_model_online)
# 上传合并后的16bit模型
model.push_to_hub_merged(new_model_online, tokenizer, save_method = "merged_16bit")
